# Step 7 — Error Analysis

Plan `docs/PLAN.md` Step 7, acceptance criteria AC-16, AC-17.

Pre-registered champion: **xgb/balanced** (`artifacts/preregistration.json`), declared on
validation before any test scoring. Test set: 56,962 rows, 98 frauds, EUR10,644.93 of
fraud value. `V1`-`V28` are anonymised PCA components -- see the caveat in Section 5.

All numbers below are post-processing on the cached arrays in `artifacts/` (AC-24); the
one exception is Section 3 (AC-17), where train-set scores were never persisted and a
refit of the three pre-registered champions is genuinely required.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
C_REVIEW = 3.0

# Anchor to the project root via the importable module, so this works
# whether the kernel cwd is /work or /work/notebooks (mirrors 01_eda.ipynb).
import preprocessing
ROOT = Path(preprocessing.__file__).resolve().parent
ART = ROOT / "artifacts"

prereg = json.loads((ART / "preregistration.json").read_text())
champion_family = prereg["champions"][0]["family"]
champion_arm = prereg["champions"][0]["arm"]
champion_key = f"{champion_family}/{champion_arm}"
champion_threshold = prereg["champions"][0]["threshold"]

val_data = np.load(ART / "val_probabilities.npz")
test_data = np.load(ART / "test_probabilities.npz")
val_results = pd.read_csv(ART / "validation_results.csv")
test_results = pd.read_csv(ART / "test_results.csv")

y_test = test_data["y"]
amt_test = test_data["amounts"]

print(f"champion  : {champion_key}  (threshold={champion_threshold:.6f}, "
      f"selected on validation per AC-24 pre-registration)")
print(f"test set  : {len(y_test):,} rows, {int(y_test.sum())} frauds, "
      f"EUR{amt_test[y_test == 1].sum():,.2f} of fraud value")

## AC-16 — missed frauds by Amount decile (headline chart)

The objective is `TotalCost = c_review * (TP + FP) + sum(Amount over FN)`. Missing a
EUR2 fraud costs the report almost nothing; missing a EUR2,000 fraud costs EUR2,000. A
plain false-negative *count* hides this completely -- this section replaces "how many
frauds did we miss" with "how much did missing them cost", broken out by Amount
decile.

In [ ]:
p_test = test_data[champion_key]
pred_test = (p_test >= champion_threshold).astype(int)

fraud_mask = y_test == 1
fraud_amt = amt_test[fraud_mask]
fraud_caught = pred_test[fraud_mask] == 1   # TP
fraud_missed = ~fraud_caught                # FN

n_fraud = int(fraud_mask.sum())
n_caught = int(fraud_caught.sum())
n_missed = int(fraud_missed.sum())
value_caught = float(fraud_amt[fraud_caught].sum())
value_missed = float(fraud_amt[fraud_missed].sum())
total_fraud_value = float(fraud_amt.sum())

assert abs(total_fraud_value - 10644.93) < 1.0, total_fraud_value

print(f"champion ({champion_key}) at threshold {champion_threshold:.6f}:")
print(f"  frauds       : {n_fraud}")
print(f"  caught (TP)  : {n_caught}  ({n_caught/n_fraud*100:.1f}% of frauds)")
print(f"  missed (FN)  : {n_missed}  ({n_missed/n_fraud*100:.1f}% of frauds)")
print(f"  value caught : EUR{value_caught:,.2f}  ({value_caught/total_fraud_value*100:.1f}% of fraud value)")
print(f"  value missed : EUR{value_missed:,.2f}  ({value_missed/total_fraud_value*100:.1f}% of fraud value)")

In [ ]:
# Decile by Amount within the 98 test frauds. rank(method="first") breaks ties
# so qcut always produces 10 equal-count bins even with repeated Amount values.
decile_df = pd.DataFrame({"amount": fraud_amt, "missed": fraud_missed.astype(int)})
decile_df["decile"] = pd.qcut(decile_df["amount"].rank(method="first"), 10, labels=False) + 1
decile_df["missed_value"] = decile_df["amount"] * decile_df["missed"]
decile_df["caught_value"] = decile_df["amount"] * (1 - decile_df["missed"])

agg = decile_df.groupby("decile").agg(
    n=("amount", "size"),
    n_missed=("missed", "sum"),
    amount_min=("amount", "min"),
    amount_max=("amount", "max"),
    value_missed=("missed_value", "sum"),
    value_caught=("caught_value", "sum"),
)
agg["n_caught"] = agg["n"] - agg["n_missed"]
agg["miss_rate_%"] = (agg["n_missed"] / agg["n"] * 100).round(1)
agg["value_total"] = agg["value_missed"] + agg["value_caught"]
display(agg.round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = agg.index.astype(str)

ax[0].bar(x, agg["n_caught"], color="#4C78A8", label="caught (TP)")
ax[0].bar(x, agg["n_missed"], bottom=agg["n_caught"], color="#E45756", label="missed (FN)")
ax[0].set_xlabel("Amount decile (1 = cheapest, 10 = priciest)")
ax[0].set_ylabel("frauds")
ax[0].set_title("Miss RATE by Amount decile")
ax[0].legend()

ax[1].bar(x, agg["value_caught"], color="#4C78A8", label="value caught")
ax[1].bar(x, agg["value_missed"], bottom=agg["value_caught"], color="#E45756", label="value missed")
ax[1].set_xlabel("Amount decile (1 = cheapest, 10 = priciest)")
ax[1].set_ylabel("EUR")
ax[1].set_title("EUR lost by Amount decile -- the cost that actually matters")
ax[1].legend()
plt.tight_layout(); plt.show()

worst_decile = int(agg["value_missed"].idxmax())
print(f"decile with the most lost EUR: decile {worst_decile} "
      f"(EUR{agg.loc[worst_decile, 'amount_min']:.2f}-{agg.loc[worst_decile, 'amount_max']:.2f}), "
      f"losing EUR{agg.loc[worst_decile, 'value_missed']:,.2f}")
print(f"decile with the highest MISS RATE: decile {int(agg['miss_rate_%'].idxmax())} "
      f"at {agg['miss_rate_%'].max():.0f}%")

**Reading the two panels together is the point.** The left panel (counts) can make the
model look worse than it is if misses cluster in the cheap deciles -- a cost objective
does not care. The right panel (EUR) is the one that maps onto `TotalCost`: a decile can
have a 100% miss rate and still be nearly irrelevant to cost if its frauds are worth a
few euros each, and a single missed fraud in the top decile can outweigh every miss in
the bottom nine deciles combined. State the actual pattern (which decile dominates
`value_missed`) using the numbers printed above when presenting this slide.

## Caught vs missed fraud value

Restating the same split in the form the rubric question ("phan tich loi") expects: of
the EUR10,644.93 of fraud in the test set, how much does the champion's alerting policy
actually catch versus let through. "Caught" here means TP under the cost model's own
definition -- flagged before the loss occurs -- not money literally returned.

In [ ]:
recovered_pct = value_caught / total_fraud_value * 100
lost_pct = value_missed / total_fraud_value * 100

print(f"total fraud value (test)     : EUR{total_fraud_value:>10,.2f}")
print(f"recovered (caught, TP)       : EUR{value_caught:>10,.2f}  ({recovered_pct:.1f}%)")
print(f"lost (missed, FN)            : EUR{value_missed:>10,.2f}  ({lost_pct:.1f}%)")
print()
print(f"cost paid in reviews (TP+FP) : EUR{C_REVIEW * pred_test.sum():>10,.2f}  "
      f"({int(pred_test.sum())} alerts x EUR{C_REVIEW:.0f})")
print(f"champion test cost           : EUR{test_results.loc[test_results.family==champion_family, 'test_cost_at_val_threshold'].iloc[0]:>10,.2f}"
      f"  (matches docs/RESULTS.md)")

## AC-17 — train vs validation: overfitting diagnosis

**T1 caveat (must be stated, not skipped).** Under the T1 branch, the decision
threshold `t*` is selected on the same validation split used for this comparison
(AC-11a). Validation Precision/Recall/F1/cost *at* `t*` are therefore optimistically
biased -- the threshold was chosen to look good on exactly this data. PR-AUC is
threshold-independent (it summarises the whole precision-recall curve, not one operating
point), so it is not subject to that bias and is the correct metric for judging
over/underfitting here. Cost-at-`t*` is shown alongside for context only, clearly
labelled as biased.

Train-set scores were never persisted (AC-24 only caches validation/test probability
vectors), so this section refits the three pre-registered champions on the training
split using the exact recipe from `run_model_matrix.py` (same features, same
`random_state=42`, same split). This is the one place in this notebook that refits
rather than loading a cached array. Refit validation PR-AUC is cross-checked against the
cached value as a correctness check on the refit itself; XGBoost's `n_jobs=-1` is not
bit-for-bit reproducible across runs (a known, documented nondeterminism -- see plan
verification step 10), so a small refit-vs-cached gap for `xgb/balanced` is expected and
not a bug.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from preprocessing import cyclic_encode_hour, hour_of_day, stratified_split_60_20_20

SEED = 42
DATA = ROOT / "data" / "creditcard.csv"

df = pd.read_csv(DATA)
y_full = df.Class.values
hours = hour_of_day(df.Time.values)
X = np.column_stack([
    df[[f"V{i}" for i in range(1, 29)]].values,
    np.log1p(df.Amount.values),
    cyclic_encode_hour(hours),
])
feature_names = [f"V{i}" for i in range(1, 29)] + ["log1p_Amount", "hour_sin", "hour_cos"]

train_idx, val_idx, test_idx = stratified_split_60_20_20(y_full, random_state=SEED)

scaler = StandardScaler().fit(X[train_idx])
assert scaler.n_samples_seen_ == len(train_idx), "scaler saw more than train -- leakage"
Xtr, Xva = scaler.transform(X[train_idx]), scaler.transform(X[val_idx])
ytr, yva = y_full[train_idx], y_full[val_idx]

pos_weight = (ytr == 0).sum() / (ytr == 1).sum()


def build(family, arm):
    """Same recipe as run_model_matrix.py's build() -- AC-25 random_state everywhere."""
    if family == "logreg":
        return LogisticRegression(
            max_iter=1000, random_state=SEED,
            class_weight="balanced" if arm == "balanced" else None)
    if family == "rf":
        return RandomForestClassifier(
            n_estimators=100, n_jobs=-1, random_state=SEED,
            class_weight="balanced" if arm == "balanced" else None)
    if family == "xgb":
        return XGBClassifier(
            tree_method="hist", eval_metric="aucpr", random_state=SEED, n_jobs=-1,
            scale_pos_weight=pos_weight if arm == "balanced" else 1.0)
    raise ValueError(family)


fitted_models = {}
rows = []
for c in prereg["champions"]:
    family, arm = c["family"], c["arm"]
    key = f"{family}/{arm}"
    model = build(family, arm).fit(Xtr, ytr)
    fitted_models[key] = model

    p_tr = model.predict_proba(Xtr)[:, 1]
    p_va = model.predict_proba(Xva)[:, 1]

    pr_auc_train = average_precision_score(ytr, p_tr)
    pr_auc_val_refit = average_precision_score(yva, p_va)
    pr_auc_val_cached = float(val_results.loc[
        (val_results.family == family) & (val_results.arm == arm), "pr_auc"
    ].iloc[0])
    val_cost_cached = float(val_results.loc[
        (val_results.family == family) & (val_results.arm == arm), "cost"
    ].iloc[0])

    rows.append({
        "model": key,
        "train_pr_auc": pr_auc_train,
        "val_pr_auc_refit": pr_auc_val_refit,
        "val_pr_auc_cached": pr_auc_val_cached,
        "refit_vs_cached_gap": pr_auc_val_refit - pr_auc_val_cached,
        "train_val_gap": pr_auc_train - pr_auc_val_refit,
        "val_cost_at_t*_BIASED": val_cost_cached,
    })

overfit_table = pd.DataFrame(rows).sort_values("train_val_gap", ascending=False)
display(overfit_table.round(4))

In [ ]:
for _, r in overfit_table.iterrows():
    print(f"{r['model']:<16} train PR-AUC {r['train_pr_auc']:.4f}  "
          f"val PR-AUC {r['val_pr_auc_refit']:.4f}  gap {r['train_val_gap']:+.4f}")

biggest = overfit_table.iloc[0]
smallest = overfit_table.iloc[-1]
print()
print(f"largest train-val PR-AUC gap  : {biggest['model']} ({biggest['train_val_gap']:+.4f})")
print(f"smallest train-val PR-AUC gap : {smallest['model']} ({smallest['train_val_gap']:+.4f})")
print()
print("refit-vs-cached validation PR-AUC gap (should be ~0 for logreg/rf, small for xgb):")
print(overfit_table[["model", "refit_vs_cached_gap"]].round(5).to_string(index=False))

**Interpretation.** A large gap between train and validation PR-AUC means the model is
fitting patterns in the training split that do not generalise -- classic overfitting. A
small gap with *both* numbers low means the model cannot separate the classes well even
on data it has memorised -- underfitting. Use the printed gaps above, not the biased
validation cost column, to defend this slide: "Random Forest reaches near-perfect
training PR-AUC because unpruned trees of depth up to their default limit can isolate
individual fraud rows; the gap to validation quantifies how much of that is
memorisation" is the shape of a 15-minute-defensible answer -- fill in the actual numbers
from the table above.

## Which frauds does the model confuse? False positives by Amount

The task is binary (fraud vs legit), so there is no multi-class confusion matrix to
inspect -- the only question available is *which legitimate transactions* the champion
mistakes for fraud. If false positives skew toward high-Amount transactions, the review
workload is concentrated on the transactions customers most want processed promptly; if
they skew low, the friction cost falls mostly on small purchases.

In [ ]:
legit_mask = y_test == 0
fp_mask = legit_mask & (pred_test == 1)
tn_mask = legit_mask & (pred_test == 0)

n_fp = int(fp_mask.sum())
n_legit = int(legit_mask.sum())

print(f"false positives       : {n_fp}  of {n_legit:,} legitimate test transactions "
      f"({n_fp/n_legit*100:.3f}%)")
print()
print(f"FP Amount    -- mean EUR{amt_test[fp_mask].mean():>8.2f}   median EUR{np.median(amt_test[fp_mask]):>8.2f}")
print(f"all legit    -- mean EUR{amt_test[legit_mask].mean():>8.2f}   median EUR{np.median(amt_test[legit_mask]):>8.2f}")
print(f"true fraud   -- mean EUR{fraud_amt.mean():>8.2f}   median EUR{np.median(fraud_amt):>8.2f}")

fp_percentile = (amt_test[legit_mask] < np.median(amt_test[fp_mask])).mean() * 100
print()
print(f"median FP Amount sits at the {fp_percentile:.0f}th percentile of the legit Amount distribution")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.logspace(0, np.log10(max(amt_test[legit_mask].max(), 1)), 50)
ax.hist(amt_test[legit_mask][amt_test[legit_mask] > 0], bins=bins, density=True,
        alpha=.5, color="#4C78A8", label="all legit transactions")
ax.hist(amt_test[fp_mask][amt_test[fp_mask] > 0], bins=bins, density=True,
        alpha=.6, color="#F58518", label="false positives")
ax.set_xscale("log")
ax.set_xlabel("Amount (EUR, log scale)")
ax.set_title(f"Where false positives sit in the legit Amount distribution ({champion_key})")
ax.legend()
plt.tight_layout(); plt.show()

**Interpretation.** Compare the FP mean/median Amount against the legit population's
mean/median printed above. Policy E-style reasoning (`p * Amount > c_review`) predicts
that *if* the champion's probabilities correlate at all with Amount, false positives
should skew toward higher Amounts, since a smaller predicted probability still clears
the bar for an expensive transaction. The champion here is scored at the AC-11a global
threshold (Policy A), not Policy E, so this is an empirical question rather than a
guaranteed consequence -- read the direction off the numbers above rather than assuming
it.

## Feature importance for the champion

**Caveat, stated up front: `V1`-`V28` are anonymised PCA components** (per the dataset
documentation and `docs/PLAN.md`). Their importance ranking is reportable -- it tells us
*which components* the champion leans on -- but it is not interpretable in business
terms, because a PCA axis is a fixed linear mixture of the original (undisclosed)
transaction features. Resist the temptation to name what "V14" means; the honest answer
in a Q&A is "this is an anonymised principal component; we can report that it matters,
not what it represents".

In [ ]:
champion_model = fitted_models[champion_key]
# .importance_type is often left at its default (None); the property that
# actually computes feature_importances_ falls back to "gain" for a tree
# booster (or "weight" for a linear booster) in that case -- report the
# EFFECTIVE type, not the possibly-None attribute.
effective_importance_type = champion_model.importance_type or (
    "weight" if champion_model.booster == "gblinear" else "gain"
)
print(f"XGBoost importance_type (attribute) = {champion_model.importance_type!r}")
print(f"XGBoost importance_type (effective) = {effective_importance_type!r}")

importances = champion_model.feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)
display(imp_df.head(15))

top_share = imp_df.head(5)["importance"].sum() / imp_df["importance"].sum() * 100
print(f"top-5 features account for {top_share:.0f}% of total importance")
print(f"log1p_Amount rank: {int(imp_df.index[imp_df.feature == 'log1p_Amount'][0]) + 1} of {len(imp_df)}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
top = imp_df.head(15).iloc[::-1]
ax.barh(top["feature"], top["importance"], color="#4C78A8")
ax.set_xlabel(f"importance ({effective_importance_type})")
ax.set_title(f"Feature importance -- {champion_key} (top 15 of {len(imp_df)})")
plt.tight_layout(); plt.show()


## Summary for the presentation

Three numbers a student should be able to defend in 15 minutes, drawn from this
notebook:

1. **AC-16 headline** -- the decile with the largest lost-EUR bar above is where the
   champion's errors actually cost money; a high miss *rate* in a cheap decile is not
   the same problem and should not be presented as one.
2. **AC-17** -- the train-val PR-AUC gap (not validation cost, which is biased by
   AC-11a's own threshold selection) ranks the three champions by overfitting risk.
3. **False positives and feature importance** -- describe *what* the champion keys on
   (which Amount range gets flagged in error, which PCA components dominate) without
   ever claiming to know what an anonymised `V`-component "means" in business terms.